In [ ]:
# Setup: clone or pull logseer repo and import core library
import os, sys
if not os.path.exists('logseer'):
    !git clone https://github.com/masahiko-shibata/logseer.git
else:
    !cd logseer && git pull
sys.path.insert(0, 'logseer')
from logseer import *
import keras

In [ ]:
# Load log data from Google Drive
from google.colab import drive
import shutil, zipfile
drive.mount('/content/drive')

!rm -rf logs data.zip 2>/dev/null
shutil.copy('/content/drive/MyDrive/Colab Notebooks/logseer/data/data.zip', 'data.zip')
with zipfile.ZipFile('data.zip', 'r') as z:
    z.extractall('.')
print(os.listdir('./'))

In [ ]:
# Configuration
import numpy as np
import pickle
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from keras.callbacks import ModelCheckpoint, EarlyStopping
from keras.models import load_model
from keras import optimizers
import xgboost as xgb
from sklearn import svm
from sklearn.ensemble import RandomForestClassifier
from scipy.stats import binomtest

BASE_DIR            = './'
DATA_DIR            = BASE_DIR + 'data'
MAX_NB_WORDS        = 24000
MAX_SEQUENCE_LENGTH = 26000
EMBEDDING_DIM       = 32
SUCCESS_LOG_RATIO   = 99
VALIDATION_SPLIT    = 0.1
VALIDATE_ON_TEST_DATA = False
EPOCHS              = 60
BATCH_SIZE          = 32
MODEL_SAVE_PATH     = 'logseer.keras'
TOKENIZER_PATH      = 'tokenizer.pickle'
EMBEDDING_LAYER     = 'vanilla'
MODEL_NAME          = 'LogCNNLite'
REPETITION          = 100
NN_ERROR_WEIGHT     = 1
ERROR_WEIGHT        = 2
LEARNING_RATE       = 0.0003
MAX_LOSS            = 0.7
PATIENCE            = 15
RESTORE_BEST_WEIGHTS = False
START_FROM_EPOCH    = 20
MONITOR             = 'val_recall'
MODE                = 'max'
RETRAIN             = False
test_NN             = True
test_xgb            = True
test_svm            = False
test_rf             = False

In [ ]:
# Training loop
ld = Loader()
tester = Tester()

for i in range(REPETITION):
    print()
    print('Repetition %s' % str(i + 1))
    texts, labels, test_texts, test_labels = ld.getdata(DATA_DIR)
    print('Found %s texts.' % len(texts))
    print('Found %s test texts.' % len(test_texts))

    train_texts, train_labels, val_texts, val_labels = split_data(
        texts, labels, test_texts, test_labels,
        validation_split=VALIDATION_SPLIT,
        validate_on_test_data=VALIDATE_ON_TEST_DATA,
    )

    tokenizer = setup_tokenizer(train_texts, TOKENIZER_PATH, MAX_NB_WORDS, retrain=RETRAIN)
    print('Found %s unique tokens in the tokenizer.' % len(tokenizer.word_index))

    if test_NN:
        train_data, val_data, test_data = prepare_sequences(
            tokenizer, train_texts, val_texts, test_texts, MAX_SEQUENCE_LENGTH)
        embedding_layer = getEmbeddingLayer(EMBEDDING_LAYER, MAX_NB_WORDS, EMBEDDING_DIM,
                                            MAX_SEQUENCE_LENGTH, word_index=tokenizer.word_index)
        ok = train_nn(
            MODEL_NAME, embedding_layer,
            train_data, train_labels, val_data, val_labels, test_data, test_labels,
            tester,
            model_save_path=MODEL_SAVE_PATH, epochs=EPOCHS, batch_size=BATCH_SIZE,
            learning_rate=LEARNING_RATE, max_loss=MAX_LOSS, error_weight=NN_ERROR_WEIGHT, retrain=RETRAIN,
        )
        if not ok:
            continue

    train_sklearn(tokenizer, train_texts, test_texts, train_labels, test_labels, tester,
                  test_xgb=test_xgb, test_svm=test_svm, test_rf=test_rf,
                  error_weight=ERROR_WEIGHT)

    if i % 10 == 9:
        tester.total(heatmap=False)

    print_ensemble(tester)

tester.total(heatmap=True)
significance_test(tester)


In [ ]:
# Save model and tokenizer to Google Drive
from google.colab import auth
from googleapiclient.http import MediaFileUpload
from googleapiclient.discovery import build

auth.authenticate_user()
drive_service = build('drive', 'v3')

def save_file_to_drive(name, path):
    file_metadata = {'name': name, 'mimeType': 'application/octet-stream'}
    media = MediaFileUpload(path, mimetype='application/octet-stream', resumable=True)
    created = drive_service.files().create(body=file_metadata, media_body=media, fields='id').execute()
    print('File ID: {}'.format(created.get('id')))
    return created

save_file_to_drive('logseer.keras', MODEL_SAVE_PATH)
save_file_to_drive('tokenizer.pickle', TOKENIZER_PATH)